# Assignment 1: Social Media Engagement Classification


In [7]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

DATA_PATH = "social_media_engagement1.csv"
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())
print("\nColumn data types:")
display(df.dtypes.to_frame("data_type"))


Dataset shape: (100, 9)


,post_id,platform,post_type,post_time,likes,comments,shares,post_day,sentiment_score
0,1,Facebook,image,8/17/2023 14:45,2121,474,628,Thursday,positive
1,2,Facebook,carousel,5/14/2023 0:45,3660,432,694,Sunday,neutral
2,3,Instagram,poll,2/21/2023 16:15,4955,408,688,Tuesday,negative
3,4,Twitter,image,11/16/2023 0:45,1183,90,187,Thursday,negative
4,5,Twitter,video,5/23/2023 0:30,3499,247,286,Tuesday,positive



Column data types:


,data_type
post_id,int64
platform,object
post_type,object
post_time,object
likes,int64
comments,int64
shares,int64
post_day,object
sentiment_score,object


In [8]:
print("Missing values per column:")
display(df.isna().sum().to_frame("missing_values"))

print("Number of duplicate rows:", df.duplicated().sum())

print("\nTarget class counts:")
display(df["sentiment_score"].value_counts().to_frame("count"))

print("\nNumerical summary:")
display(df.describe())


Missing values per column:


,missing_values
post_id,0
platform,0
post_type,0
post_time,0
likes,0
comments,0
shares,0
post_day,0
sentiment_score,0


Number of duplicate rows: 0

Target class counts:


,count
sentiment_score,
positive,46
neutral,27
negative,27



Numerical summary:


,post_id,likes,comments,shares
count,100.000000,100.000000,100.00000,100.000000
mean,50.500000,2381.810000,202.66000,415.650000
std,29.011492,1632.573284,138.84067,283.877601
min,1.000000,15.000000,10.00000,16.000000
25%,25.750000,895.750000,89.75000,183.000000
50%,50.500000,2220.000000,171.00000,356.500000
75%,75.250000,3593.250000,299.00000,689.500000
max,100.000000,5000.000000,500.00000,993.000000


## Preprocessing and feature engineering

In [10]:
df["post_time"] = pd.to_datetime(df["post_time"], errors="coerce")
df["post_hour"] = df["post_time"].dt.hour
df["post_month"] = df["post_time"].dt.month

df_model = df.drop(columns=["post_id", "post_time"])

target = "sentiment_score"
X = df_model.drop(columns=[target])
y = df_model[target]

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)
print("Target classes:", sorted(y.unique()))


Numerical features: ['likes', 'comments', 'shares', 'post_hour', 'post_month']
Categorical features: ['platform', 'post_type', 'post_day']
Target classes: ['negative', 'neutral', 'positive']


## Train-test split

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("\nTraining target distribution:")
display(y_train.value_counts(normalize=True).rename("proportion").to_frame())


Training rows: 80
Testing rows: 20

Training target distribution:


,proportion
sentiment_score,
positive,0.4625
neutral,0.2750
negative,0.2625


## Build, train, and evaluate the model

In [12]:
numeric_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_preprocessor, numeric_features),
    ("categorical", categorical_preprocessor, categorical_features)
])

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced"
    ))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy:.2%}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("Confusion Matrix (rows = actual, columns = predicted):")
labels = model.named_steps["classifier"].classes_
display(pd.DataFrame(
    confusion_matrix(y_test, y_pred, labels=labels),
    index=[f"Actual {label}" for label in labels],
    columns=[f"Predicted {label}" for label in labels]
))


Test Accuracy: 50.00%

Classification Report:
              precision    recall  f1-score   support

    negative       0.50      0.33      0.40         6
     neutral       0.50      0.20      0.29         5
    positive       0.50      0.78      0.61         9

    accuracy                           0.50        20
   macro avg       0.50      0.44      0.43        20
weighted avg       0.50      0.50      0.47        20

Confusion Matrix (rows = actual, columns = predicted):


,Predicted negative,Predicted neutral,Predicted positive
Actual negative,2,1,3
Actual neutral,0,1,4
Actual positive,2,0,7
